# Практика · Ідея передтренування

> Теорія: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ Зошит навчає **шістдесят сім** маленьких мереж: одну довго й **без міток**,
> решту — коротко й з мітками. Заміряно: **близько 250 с процесорного часу** на
> чотирьох ядрах без відеокарти, потоки зафіксовано в один. Стінний час залежить
> від того, чим ще зайнята машина: на вільній він близький до процесорного, на
> завантаженій — у півтора-два рази більший.

Задача цього зошита — одна, і вона справжня: **чи повідомляє цей рядок інтерфейсу
про помилку**. Дані — українські переклади програм, що вже стоять на цій машині.

Ми зробимо чотири речі:

1. Візьмемо **рубіж**: TF-IDF із логістичною регресією на всіх мітках.
2. Наробимо **безкоштовних міток** із самого тексту — закриємо слово й попросимо
   модель його вгадати. Це передтренування.
3. **Заморозимо** передтреновану модель і подивимось, чи є в її векторах щось
   корисне, чого немає у випадкових.
4. Побудуємо криву **«якість від кількості міток»** для двох гілок — передтренованої
   й навченої з нуля — і подивимось, де саме (і чи взагалі) передтренування виграє.


## 1 · Середовище й чесний годинник

Спільна машина заважає міряти час двома способами. По-перше, стінний годинник
показує не роботу, а очікування. По-друге — і це менш очевидно — **процесорний час
теж бреше**, якщо бібліотеки розводять роботу на кілька потоків: потоки крутяться
в очікуванні одне одного, і це очікування рахується як робота.

Тому перед імпортом `numpy` і `torch` ми фіксуємо кількість потоків в **один**,
а міряємо `time.process_time()` — процесорний час, а не настінний.


In [ ]:
import os

# ⚠️ ці рядки мусять стояти ДО імпорту numpy і torch: бібліотеки читають
# змінні оточення один раз, у момент завантаження
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
             'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

import re, glob, gettext, gc, copy, time, sys
from collections import Counter

import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)          # те саме для внутрішнього пулу torch

STARTED = time.process_time()     # звідси рахуємо процесорний час усього зошита

print('python     :', sys.version.split()[0])
print('numpy      :', np.__version__)
print('torch      :', torch.__version__)
print('потоків    :', torch.get_num_threads())


## 2 · Корпус: українські переклади з цієї машини

Кожна програма з графічним інтерфейсом носить із собою файл перекладів —
`.mo` у теці `/usr/share/locale/uk/LC_MESSAGES/`. Усередині лежать **пари**:
англійський оригінал рядка й український переклад.

Ця пара — рідкісна розкіш. Ознаки ми братимемо з **українського** боку, а мітку —
з **англійського**. Модель не бачить того боку, з якого зроблено мітку, тож замір
не буде круговим.

Одну обережність зробимо одразу: **однакові переклади трапляються в багатьох
програмах**. Якщо їх не прибрати, один і той самий рядок опиниться і в навчальній,
і в перевірній частині, і оцінка вийде завищеною. Лишаємо кожен переклад один раз.


In [ ]:
def load_corpus():
    """Читає українські .mo-файли й повертає трійки (програма, оригінал, переклад).

    Дублікати перекладів прибираємо одразу: той самий рядок у двох програмах —
    це один документ, а не два.
    """
    docs, seen = [], set()
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as handle:
                catalog = gettext.GNUTranslations(handle)
        except Exception:
            continue                      # чужий або зламаний формат — пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і текстом не є
            if not (isinstance(source, str) and isinstance(target, str)):
                continue
            if len(target) <= 30 or 'Project-Id' in target or target in seen:
                continue
            seen.add(target)
            docs.append((program, source, target))
    return docs


corpus = load_corpus()

if len(corpus) < 20000:
    raise SystemExit(
        'Українських перекладів на цій машині замало: знайдено %d документів.\n'
        'Зошитові потрібні щонайменше 20 000. Постав українську локаль\n'
        '(у Fedora: sudo dnf install glibc-langpack-uk, плюс мовні пакети програм)\n'
        'і запусти ще раз. Підставного мінікорпусу тут навмисно немає: на кількох\n'
        'сотнях рядків жоден замір цього зошита не має сенсу.' % len(corpus))

print('документів :', len(corpus))
print('програм    :', len({program for program, _, _ in corpus}))
print()
print('перші три записи:')
for program, source, target in corpus[:3]:
    print(f'  [{program}]')
    print(f'     англ: {source[:70]}')
    print(f'     укр : {target[:70]}')


## 3 · Мітка: чи повідомляє рядок про помилку

Мітку робить регулярка по **англійському** боку — вісім слів, якими англомовні
програми повідомляють про збій. Це не ідеальна розмітка, але вона однакова для
всіх моделей, тож порівняння між ними чесне.


In [ ]:
ERROR_WORDS = re.compile(
    r'\b(error|failed|cannot|could not|unable|invalid|denied|no such)\b', re.I)

sources   = [source for _, source, _ in corpus]     # англійський бік — тільки мітка
documents = [target for _, _, target in corpus]     # український бік — тільки ознаки
labels    = np.array([1 if ERROR_WORDS.search(s) else 0 for s in sources])

print('клас «помилка»    :', int(labels.sum()), f'= {labels.mean():.4f}')
print('клас «не помилка» :', int((labels == 0).sum()), f'= {1 - labels.mean():.4f}')
print()
shown = {0: 0, 1: 0}
for i in range(len(corpus)):
    y = labels[i]
    if shown[y] >= 2 or len(documents[i]) > 60:
        continue
    shown[y] += 1
    print(f'  мітка {y}: {documents[i]}')


## 4 · Токенізація, поділ і словник

Токенізатор — канонічний для курсу: пробіги українських літер, апостроф вважається
звʼязкою всередині слова, а не окремим символом.

**Поділ робимо після перемішування.** Це не формальність: корпус лежить у порядку
програм, і «перші 80 %» означали б не «більше даних», а «кілька десятків програм
замість двохсот пʼятдесяти». Перемішуємо з фіксованим зерном, щоб поділ був
однаковий у всіх, хто запустить зошит.

Три частини:

| частина | навіщо |
|---|---|
| навчальна, 80 % | передтренування (без міток) і донавчання (з мітками) |
| **відкладена**, 10 % | добір швидкості навчання — і більше нічого |
| перевірна, 10 % | остаточні числа, які ми цитуємо |


In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"
TOKENIZER = re.compile(TOKEN_PATTERN)

tokenised = [TOKENIZER.findall(text.lower()) for text in documents]

lengths = np.array([len(t) for t in tokenised])
print('слововживань      :', int(lengths.sum()))
print('довжина: медіана %.0f, середня %.2f, 90-й процентиль %.0f'
      % (np.median(lengths), lengths.mean(), np.percentile(lengths, 90)))

# ⚠️ корпус лежить у порядку програм — ріжемо тільки після перемішування
shuffler = np.random.default_rng(0)
order = shuffler.permutation(len(corpus))
n_all = len(order)
train_idx = order[:int(0.8 * n_all)]
dev_idx   = order[int(0.8 * n_all):int(0.9 * n_all)]
test_idx  = order[int(0.9 * n_all):]

print()
print('навчальна  :', len(train_idx))
print('відкладена :', len(dev_idx))
print('перевірна  :', len(test_idx))


Словник будуємо **тільки з навчальної частини** — інакше модель знала б про слова,
яких у навчанні не бачила, і перевірка стала б трохи фіктивною.

Чотири службові позиції йдуть першими:

| позиція | що означає |
|---|---|
| `[PAD]` | добивка до однакової довжини, модель її ігнорує |
| `[UNK]` | слово, якого немає в словнику |
| `[MASK]` | тут стояло слово, і його треба вгадати |
| `[CLS]` | службовий початок рядка |


In [ ]:
PAD, UNK, MASK, CLS = 0, 1, 2, 3
MAX_LEN = 12                     # 90 % рядків коротші — довший хвіст обрізаємо

counter = Counter(word for i in train_idx for word in tokenised[i])
vocabulary = {'[PAD]': PAD, '[UNK]': UNK, '[MASK]': MASK, '[CLS]': CLS}
for word, count in counter.most_common():
    if count >= 10:              # рідкісні слова моделі однаково не вивчити
        vocabulary[word] = len(vocabulary)

VOCAB_SIZE = len(vocabulary)
inverse_vocabulary = {index: word for word, index in vocabulary.items()}


def encode(token_lists):
    """Кожен рядок → [CLS] і не більше ніж MAX_LEN-1 номерів слів, решта — [PAD]."""
    matrix = np.zeros((len(token_lists), MAX_LEN), dtype=np.int64)
    for row, tokens in enumerate(token_lists):
        ids = [CLS] + [vocabulary.get(word, UNK) for word in tokens[:MAX_LEN - 1]]
        matrix[row, :len(ids)] = ids
    return matrix


ids_all = encode(tokenised)
ids_train, ids_dev, ids_test = ids_all[train_idx], ids_all[dev_idx], ids_all[test_idx]
y_train,   y_dev,   y_test   = labels[train_idx], labels[dev_idx], labels[test_idx]

real = ids_train[ids_train != PAD]
print('словник (слово трапилось ≥ 10 разів):', VOCAB_SIZE)
print('частка [UNK] серед реальних позицій :', f'{(real == UNK).mean():.4f}')


## 5 · Рубіж: TF-IDF із логістичною регресією

Перш ніж будувати щось складне, треба знати, що дає просте. TF-IDF рахує вагу
кожного слова в документі, логістична регресія проводить по цих вагах межу.
Жодного навчання подань, жодних нейромереж — і **всі мітки корпусу**.

Дивимось на F1 по класу «помилка»: клас нерівний (приблизно один до чотирьох),
тож точність тут ні про що не скаже.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


def f1_error(y_true, y_pred):
    """F1 саме по класу «помилка». Пишемо руками, щоб було видно, з чого складається."""
    true_positive  = int(((y_pred == 1) & (y_true == 1)).sum())
    false_positive = int(((y_pred == 1) & (y_true == 0)).sum())
    false_negative = int(((y_pred == 0) & (y_true == 1)).sum())
    if true_positive == 0:
        return 0.0
    precision = true_positive / (true_positive + false_positive)
    recall    = true_positive / (true_positive + false_negative)
    return 2 * precision * recall / (precision + recall)


joined = [' '.join(tokens) for tokens in tokenised]
baseline_scores = []
for seed in range(3):
    rng = np.random.default_rng(seed)
    split = rng.permutation(n_all)
    fit_part, check_part = split[:int(0.8 * n_all)], split[int(0.9 * n_all):]
    vectorizer = TfidfVectorizer(token_pattern=r'(?u)\S+', min_df=2)
    X_fit   = vectorizer.fit_transform([joined[i] for i in fit_part])
    X_check = vectorizer.transform([joined[i] for i in check_part])
    model = LogisticRegression(max_iter=1000).fit(X_fit, labels[fit_part])
    baseline_scores.append(f1_error(labels[check_part], model.predict(X_check)))

TFIDF_F1 = float(np.mean(baseline_scores))
print('ознак у TF-IDF :', X_fit.shape[1])
print('F1 «помилка» по зернах:', ' '.join(f'{s:.4f}' for s in baseline_scores))
print(f'РУБІЖ: {TFIDF_F1:.4f} ±{np.std(baseline_scores):.4f}')


## 6 · Звідки беруться безкоштовні мітки

Ось увесь фокус передтренування. Беремо речення, **прибираємо з нього слово** —
і маємо приклад із відповіддю. Запитання: речення з діркою. Відповідь: те, що
було в дірці. Ніхто нічого не розмічав: відповідь лежала в тексті, ми просто
вийняли її самі.

Прибирати треба **саме прибирати**. Якщо слово лишити на місці, модель прочитає
відповідь із входу, втрата впаде майже до нуля — і не навчиться нічого. До цього
ми ще повернемось окремим заміром.

Схема псування — та сама, що в BERT. Беремо 15 % позицій, і з них:

| частка | що робимо | навіщо |
|---|---|---|
| 80 % | ставимо `[MASK]` | основний випадок |
| 10 % | ставимо **випадкове** слово | щоб модель не вірила входу сліпо |
| 10 % | лишаємо як є | щоб вона працювала і без `[MASK]` |


In [ ]:
def corrupt(batch, generator):
    """Псує батч за схемою 15 % / 80-10-10 і каже, які позиції треба вгадати."""
    corrupted = batch.clone()
    # [PAD] псувати нічого, [CLS] теж: обидва не є словами
    eligible = (batch != PAD) & (batch != CLS)
    chosen = eligible & (torch.rand(batch.shape, generator=generator) < 0.15)
    if chosen.sum() == 0:                       # виродження на крихітному батчі
        chosen[0, 1] = eligible[0, 1]
    how = torch.rand(batch.shape, generator=generator)
    corrupted[chosen & (how < 0.80)] = MASK
    random_slot = chosen & (how >= 0.80) & (how < 0.90)
    if random_slot.sum() > 0:
        corrupted[random_slot] = torch.randint(
            4, VOCAB_SIZE, (int(random_slot.sum()),), generator=generator)
    return corrupted, chosen


maskable = int(((ids_train != PAD) & (ids_train != CLS)).sum())
print('позицій, які можна закрити в навчальній частині:', maskable)
print('тобто стільки прикладів із відповідями — і жодної людської години')
print()

demo = torch.from_numpy(ids_train[:1])
demo_generator = torch.Generator().manual_seed(3)
spoiled, asked = corrupt(demo, demo_generator)
print('було  :', ' '.join(inverse_vocabulary[i] for i in demo[0].tolist() if i != PAD))
print('стало :', ' '.join(inverse_vocabulary[i] for i in spoiled[0].tolist() if i != PAD))
print('питають про позиції:', asked[0].nonzero().flatten().tolist())


## 7 · Модель: трансформерний енкодер на два шари

Це **навчальний макет**, а не BERT. Справжній BERT-base — 110 мільйонів параметрів
і тижні на кластері; у нас модель у сто разів менша, а корпус — у тисячі разів.
Механізм від цього той самий, а очікування треба тримати скромні.

Будова проста: ембединги слів плюс ембединги позицій, два шари енкодера,
нормалізація. Над цим ставиться **голова** — і саме голова визначає, яку задачу
модель розвʼязує:

- голова передтренування: `Linear(64 → словник)`, вгадує закрите слово;
- голова класифікації: `Linear(64 → 2)`, каже «помилка / не помилка».

Тіло — спільне. У цьому й уся ідея переносу.


In [ ]:
D_MODEL, N_LAYERS, N_HEADS = 64, 2, 4


class Encoder(nn.Module):
    """Тіло моделі: перетворює номери слів на вектори, що вже враховують сусідів."""

    def __init__(self):
        super().__init__()
        self.token = nn.Embedding(VOCAB_SIZE, D_MODEL, padding_idx=PAD)
        self.position = nn.Embedding(MAX_LEN, D_MODEL)
        layer = nn.TransformerEncoderLayer(
            D_MODEL, N_HEADS, 2 * D_MODEL, dropout=0.1,
            batch_first=True, norm_first=True, activation='gelu')
        self.body = nn.TransformerEncoder(layer, N_LAYERS, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(D_MODEL)

    def forward(self, ids):
        pad_mask = (ids == PAD)          # добивку увага мусить ігнорувати
        positions = torch.arange(ids.shape[1])
        hidden = self.token(ids) + self.position(positions)
        hidden = self.body(hidden, src_key_padding_mask=pad_mask)
        return self.norm(hidden), pad_mask


def mean_pool(hidden, pad_mask):
    """Один вектор на весь рядок: середнє по реальних позиціях, добивка не рахується."""
    keep = (~pad_mask).unsqueeze(-1).float()
    return (hidden * keep).sum(1) / keep.sum(1).clamp(min=1.0)


class Classifier(nn.Module):
    """Те саме тіло плюс лінійна голова на два класи."""

    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(D_MODEL, 2)

    def forward(self, ids):
        hidden, pad_mask = self.encoder(ids)
        return self.head(mean_pool(hidden, pad_mask))


probe_encoder = Encoder()
body_size = sum(p.numel() for p in probe_encoder.parameters())
print('параметрів у тілі            :', body_size)
print('  з них таблиця ембедингів   :', VOCAB_SIZE * D_MODEL)
print('голова передтренування       :', D_MODEL * VOCAB_SIZE + VOCAB_SIZE)
print('голова класифікації          :', D_MODEL * 2 + 2)
del probe_encoder
gc.collect()


## 8 · Передтренування: довго, дорого, один раз

Тепер найдовша клітинка зошита. Модель читає навчальну частину корпусу **без
жодної мітки** й учиться вгадувати закриті слова.

Швидкість навчання тут — `3e-3` із коротким розігрівом. Її дібрано окремим
прогоном по відкладеній частині: на сітці `3e-4 … 1e-2` мінімум втрати лежить
усередині, а не скраю.

Щоб знати, чи модель узагалі щось вивчила, потрібен рубіж. Найдешевший —
**уніграмний**: вгадувати завжди найчастіші слова, не дивлячись на контекст.
Його перехресна ентропія рахується прямо з частот, без жодного навчання.
Якщо втрата моделі не опустилась нижче — контексту вона не бачить.


In [ ]:
# уніграмний рубіж: скільки коштує вгадування без огляду на сусідів
positions = ids_train[(ids_train != PAD) & (ids_train != CLS)]
frequency = np.bincount(positions, minlength=VOCAB_SIZE) / len(positions)
frequency = frequency[frequency > 0]
UNIGRAM_LOSS = float(-(frequency * np.log(frequency)).sum())

print(f'рівномірне вгадування (ln {VOCAB_SIZE}) : {np.log(VOCAB_SIZE):.4f}')
print(f'уніграмний рубіж                  : {UNIGRAM_LOSS:.4f}')


In [ ]:
PRETRAIN_STEPS = 1600
PRETRAIN_BATCH = 64
PRETRAIN_LR = 3e-3


def pretrain(steps=PRETRAIN_STEPS, seed=0, report_every=400):
    """Маскована мовна модель. Міток не бачить взагалі — тільки текст."""
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed + 10_000)
    encoder = Encoder()
    head = nn.Linear(D_MODEL, VOCAB_SIZE)
    parameters = list(encoder.parameters()) + list(head.parameters())
    optimizer = torch.optim.AdamW(parameters, lr=PRETRAIN_LR, weight_decay=0.01)
    # розігрів: перші 100 кроків крок росте лінійно, інакше великий крок ламає старт
    schedule = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: min(1.0, (step + 1) / 100))

    data = torch.from_numpy(ids_train)
    history = []
    encoder.train()
    for step in range(steps):
        pick = torch.randint(0, data.shape[0], (PRETRAIN_BATCH,), generator=generator)
        original = data[pick]
        spoiled, asked = corrupt(original, generator)
        hidden, _ = encoder(spoiled)
        # рахуємо втрату ЛИШЕ на закритих позиціях — решта відповідей видно на вході
        loss = nn.functional.cross_entropy(head(hidden[asked]), original[asked])
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(parameters, 1.0)
        optimizer.step()
        schedule.step()
        history.append(float(loss.detach()))
        if report_every and (step + 1) % report_every == 0:
            print(f'   крок {step + 1:5d}   втрата {np.mean(history[-report_every:]):.4f}',
                  flush=True)

    weights = copy.deepcopy(encoder.state_dict())
    head_weights = copy.deepcopy(head.state_dict())
    del encoder, head, optimizer
    gc.collect()
    return weights, head_weights, history


clock = time.process_time()
PRETRAINED, PRETRAINED_HEAD, history = pretrain()
pretrain_seconds = time.process_time() - clock

print()
print(f'передтренування: {pretrain_seconds:.1f} c процесорних '
      f'({pretrain_seconds / PRETRAIN_STEPS:.4f} c/крок)')
print(f'втрата: перші 50 кроків {np.mean(history[:50]):.4f} '
      f'→ останні 50 {np.mean(history[-50:]):.4f}')
print(f'уніграмний рубіж {UNIGRAM_LOSS:.4f} — '
      f'{"пройдено" if np.mean(history[-50:]) < UNIGRAM_LOSS else "НЕ пройдено"}, '
      f'запас {UNIGRAM_LOSS - np.mean(history[-50:]):+.4f} ната')


## 9 · Що воно вивчило: подивимось на відповіді

Число втрати — це добре, але корисно побачити, що модель насправді пропонує.
Закриємо одне слово в справжньому рядку й попросимо пʼять найімовірніших варіантів.

Дивитись треба не на те, чи вгадано точно (з пʼяти тисяч слів це рідкість), а на
те, чи варіанти **того самого сорту**: та сама частина мови, той самий відмінок,
той самий сенс. Саме це й означає «модель бачить контекст».


In [ ]:
@torch.no_grad()
def guess(sentence_ids, position, weights, head_weights, top=5):
    """Закриває одну позицію й повертає найімовірніші слова на її місце."""
    encoder = Encoder(); encoder.load_state_dict(weights); encoder.eval()
    head = nn.Linear(D_MODEL, VOCAB_SIZE); head.load_state_dict(head_weights); head.eval()
    query = torch.from_numpy(sentence_ids.copy()).unsqueeze(0)
    answer = inverse_vocabulary[int(query[0, position])]
    query[0, position] = MASK
    hidden, _ = encoder(query)
    scores = head(hidden[0, position])
    best = torch.topk(scores, top).indices.tolist()
    del encoder, head
    gc.collect()
    return answer, [inverse_vocabulary[i] for i in best]


# беремо кілька довших рядків навчальної частини — на них контекст видно краще
long_rows = [i for i in range(len(ids_train)) if (ids_train[i] != PAD).sum() >= 8][:3]
for row in long_rows:
    words = [inverse_vocabulary[i] for i in ids_train[row] if i != PAD]
    hole = 3
    answer, options = guess(ids_train[row], hole, PRETRAINED, PRETRAINED_HEAD)
    shown = list(words)
    shown[hole] = '____'
    print(' '.join(shown[1:]))
    print(f'   було: {answer}')
    print(f'   модель пропонує: {", ".join(options)}')
    print()


## 10 · Що переноситься: заморожений зонд

Найпряміший спосіб перевірити, чи передтренування щось поклало в модель:
**заморозити її повністю** й дозволити вчитися лише одному лінійному шару зверху.
Такий шар нічого не вміє сам — він може тільки провести межу в тому просторі,
який йому дали. Це називають **лінійним зондом**.

Порівнюємо два простори однакового розміру: від передтренованого тіла й від
такого самого тіла з **випадковими** вагами. Різниця між ними — і є те, що
поклало туди передтренування.


In [ ]:
@torch.no_grad()
def embed(weights, ids, batch=512):
    """Проганяє рядки через заморожене тіло й повертає по одному вектору на рядок."""
    encoder = Encoder()
    if weights is not None:
        encoder.load_state_dict(weights)
    encoder.eval()
    data = torch.from_numpy(ids)
    chunks = []
    for start in range(0, len(ids), batch):
        hidden, pad_mask = encoder(data[start:start + batch])
        chunks.append(mean_pool(hidden, pad_mask).numpy())
    del encoder
    gc.collect()
    return np.vstack(chunks)


torch.manual_seed(0)
probe_pool = train_idx[:3000]                      # частина, на якій учиться зонд
probe_ids = ids_all[probe_pool]
probe_y = labels[probe_pool]

print(f'{"простір":<28}{"F1 «помилка»":>16}')
probe_result = {}
for name, weights in (('випадкове тіло', None), ('передтреноване тіло', PRETRAINED)):
    train_vectors = embed(weights, probe_ids)
    test_vectors = embed(weights, ids_test)
    linear = LogisticRegression(max_iter=1000).fit(train_vectors, probe_y)
    probe_result[name] = f1_error(y_test, linear.predict(test_vectors))
    print(f'{name:<28}{probe_result[name]:>16.4f}')

print()
print(f'різниця: {probe_result["передтреноване тіло"] - probe_result["випадкове тіло"]:+.4f}')
print('обидві гілки бачили однакові 3000 міток і мали однакову лінійну голову —')
print('різниця цілком у тому, який простір їм дали')


## 11 · Донавчання: правила чесного порівняння

Далі — головний замір. Дві гілки:

- **передтренована**: тіло стартує з ваг, здобутих у розділі 8;
- **з нуля**: те саме тіло, але ваги випадкові.

Усе інше однакове: та сама архітектура, той самий набір міток, той самий батч,
та сама кількість кроків, той самий обрізач градієнта. Різниця рівно в стартових вагах.

Два запобіжники, без яких порівняння було б нечесним:

1. **Швидкість навчання добираємо кожній гілці окремо** — і на **відкладеній**
   частині, не на перевірній. Інакше ми або зіпсуємо одну з гілок чужим кроком,
   або підглянемо у відповідь.
2. **Підвибірку міток беремо випадково**, а не «перші n». Перші n — це кілька
   програм, а не менше даних.


In [ ]:
FINETUNE_BATCH = 32


def steps_for(n_labels):
    """Бюджет донавчання: приблизно чотири проходи по мітках, але від 80 до 200 кроків."""
    return int(min(200, max(80, round(4 * n_labels / FINETUNE_BATCH))))


def finetune(n_labels, weights, seed, lr):
    """Донавчає модель на n_labels мітках. weights=None — гілка «з нуля»."""
    picker = np.random.default_rng(seed)
    picked = picker.choice(len(ids_train), size=n_labels, replace=False)
    x = torch.from_numpy(ids_train[picked])
    y = torch.from_numpy(y_train[picked])

    torch.manual_seed(seed)
    encoder = Encoder()
    if weights is not None:
        encoder.load_state_dict(weights)
    model = Classifier(encoder)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    generator = torch.Generator().manual_seed(seed + 777)

    model.train()
    for _ in range(steps_for(n_labels)):
        batch = torch.randint(0, n_labels, (min(FINETUNE_BATCH, n_labels),),
                              generator=generator)
        loss = nn.functional.cross_entropy(model(x[batch]), y[batch])
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    return model


@torch.no_grad()
def predict(model, ids, batch=512):
    model.eval()
    data = torch.from_numpy(ids)
    out = []
    for start in range(0, len(ids), batch):
        out.append(model(data[start:start + batch]).argmax(1).numpy())
    return np.concatenate(out)


for n in (200, 800, 3200):
    print(f'{n:>5} міток → {steps_for(n):>3} кроків '
          f'({steps_for(n) * FINETUNE_BATCH / n:.1f} проходів по набору)')


Тепер сітка швидкості навчання. Шість значень, логарифмічна сітка, **обидві гілки**,
**кожен бюджет міток окремо**, оцінка — на **відкладеній** частині.

Три вимоги, і жодну не можна пропустити:

- **сітка широка**: якщо найкраще значення опиняється **скраю**, ти знайшов не
  оптимум, а межу перебору — і число нічого не варте;
- **добираємо тим бюджетом, з яким працюватимемо**: короткий бюджет систематично
  тягне до більшого кроку, тож крок, дібраний на іншій кількості кроків, не той;
- **обом гілкам однаково**: та сама сітка, та сама вибірка, той самий бюджет.


In [ ]:
LR_GRID = [3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
BUDGETS = [200, 800, 3200]

best_lr = {}
print(f'{"гілка":<16}{"міток":>6}' + ''.join(f'{lr:>9.0e}' for lr in LR_GRID) + '   вибір')
for n_labels in BUDGETS:
    for branch, weights in (('передтренована', PRETRAINED), ('з нуля', None)):
        row = []
        for lr in LR_GRID:
            model = finetune(n_labels, weights, seed=100, lr=lr)
            row.append(f1_error(y_dev, predict(model, ids_dev)))
            del model
            gc.collect()
        winner = int(np.argmax(row))
        best_lr[(branch, n_labels)] = LR_GRID[winner]
        edge = ' ← КРАЙ СІТКИ' if winner in (0, len(LR_GRID) - 1) else ''
        print(f'{branch:<16}{n_labels:>6}' + ''.join(f'{v:>9.4f}' for v in row)
              + f'   {LR_GRID[winner]:.0e}{edge}', flush=True)


## 12 · Головна крива: якість від кількості міток

Три бюджети міток — 200, 800, 3200 — і пʼять зерен на кожну точку. Кожна гілка
йде зі своїм кроком, дібраним щойно саме для цього бюджету. Зерно міняє
**і** підвибірку міток, **і** випадкову ініціалізацію, **і** порядок батчів, тож
розкид по зернах чесно показує, наскільки результату можна вірити.

Порівнюємо не середні, а **купи**: якщо найгірший результат однієї гілки вищий
за найкращий результат другої, різниця є. Якщо купи перетинаються — різниці немає,
хоч би як розходились середні.


In [ ]:
SEEDS = range(5)

curve = {}
print(f'{"гілка":<18}{"міток":>7}{"медіана":>10}{"середнє":>10}{"розкид":>9}'
      f'{"купа":>19}')
for n_labels in BUDGETS:
    for branch, weights in (('передтренована', PRETRAINED), ('з нуля', None)):
        scores = []
        for seed in SEEDS:
            model = finetune(n_labels, weights, seed=seed, lr=best_lr[(branch, n_labels)])
            scores.append(f1_error(y_test, predict(model, ids_test)))
            del model
            gc.collect()
        curve[(branch, n_labels)] = scores
        print(f'{branch:<18}{n_labels:>7}{np.median(scores):>10.4f}'
              f'{np.mean(scores):>10.4f}{np.std(scores):>9.4f}'
              f'{min(scores):>10.4f}…{max(scores):<8.4f}', flush=True)


## 13 · Читаємо результат

Три питання до таблиці:

1. Чи виграє передтренована гілка — і **на яких** бюджетах?
2. Чи **перетинаються купи**? Різниця, менша за розкид, різницею не є.
3. Чи дотягується хоч одна гілка до рубежу TF-IDF?


In [ ]:
print(f'{"міток":>7}{"передтренована":>17}{"з нуля":>11}{"різниця":>11}   висновок')
for n_labels in BUDGETS:
    warm = curve[('передтренована', n_labels)]
    cold = curve[('з нуля', n_labels)]
    separated = min(warm) > max(cold) or min(cold) > max(warm)
    verdict = 'купи НЕ перетинаються' if separated else 'купи перетинаються — різниці немає'
    print(f'{n_labels:>7}{np.median(warm):>17.4f}{np.median(cold):>11.4f}'
          f'{np.median(warm) - np.median(cold):>+11.4f}   {verdict}')

best_overall = max(np.median(curve[k]) for k in curve)
print()
print(f'рубіж TF-IDF на ВСІХ мітках     : {TFIDF_F1:.4f}')
print(f'найкраща точка кривої           : {best_overall:.4f}')
print(f'відставання                     : {best_overall - TFIDF_F1:+.4f}')


## 14 · Скільки це коштувало

Остання клітинка — рахунок. Передтренування платиться **один раз**, донавчання —
щоразу. Саме тому перше вважають вигідним: його вартість ділиться на кількість
задач, а виграш у мітках лишається кожній.


In [ ]:
finetune_cost = pretrain_seconds / PRETRAIN_STEPS  # приблизна ціна кроку
total = time.process_time() - STARTED

print(f'передтренування ({PRETRAIN_STEPS} кроків) : {pretrain_seconds:>7.1f} c')
print(f'увесь зошит                        : {total:>7.1f} c процесорних')
print()
print('амортизація одного передтренування по задачах:')
one_finetune = pretrain_seconds / PRETRAIN_STEPS * steps_for(800) * 0.5
for tasks in (1, 5, 20, 100):
    share = pretrain_seconds / tasks
    print(f'  {tasks:>4} задач → на задачу припадає {share:>7.1f} c передтренування')


## Домашнє завдання

Повний текст із критеріями — у [homework.html](homework.html). Коротко:

**🟢 Рівень 1.** Додай до кривої четвертий бюджет — 12 800 міток — і скажи, чи
різниця між гілками там зникає остаточно. Не забудь: крок дібрано на 800 мітках,
тож перевір, чи він не зсунувся.

**🟡 Рівень 2.** Заміряй, як виграш залежить від **кількості кроків
передтренування**: 0, 550, 1100, 2200. Побудуй криву F1 при 200 мітках.

**🔴 Рівень 3.** Зроби «зіпсовану» проксі-задачу: передтренуй модель без псування
входу — тобто попроси її вгадати слово, яке видно на вході. Покажи числом, що
втрата падає майже до нуля, а перенесення не дає нічого.
